In [1]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import ListedColormap
from PIL import Image
from pytorch_msssim import MS_SSIM

import afmlevel.afmlevel_functions as afml

In [2]:
def znorm(image):
    mean = np.mean(image)
    std_dev = np.std(image)
    imnorm_z = (image - mean) / std_dev
    return imnorm_z


def minmaxnorm(image):
    if isinstance(image, torch.Tensor):
        im0 = image - image.min()
        imnorm = im0 / im0.max()
        return imnorm
    elif isinstance(image, np.ndarray):
        im0 = image - np.min(image)
        imnorm = im0 / np.max(im0)
        return imnorm


def xyplanefit(imarray, polyx, polyy):
    mc = np.mean(imarray, axis=0)
    x = np.arange(0, len(mc), 1)
    p = np.polyfit(x, mc, polyx)
    p = np.poly1d(p)
    pvals = np.polyval(p, x)
    r = imarray - pvals[np.newaxis, :]

    mr = np.mean(r, axis=1)
    y = np.arange(0, len(mr), 1)
    p = np.polyfit(y, mr, polyy)
    p = np.poly1d(p)
    pvals = np.polyval(p, y)
    r = r - pvals[:, np.newaxis]

    return r


def MS_SSIMloss(predicted, groundtruth, device="cpu"):
    if not isinstance(predicted, torch.Tensor):
        predicted = torch.from_numpy(predicted)
    if not isinstance(groundtruth, torch.Tensor):
        groundtruth = torch.from_numpy(groundtruth)

    predicted = predicted.float().to(device)
    groundtruth = groundtruth.float().to(device)
    predicted = predicted.unsqueeze(0).unsqueeze(0)
    groundtruth = groundtruth.unsqueeze(0).unsqueeze(0)

    MS_SSIMfn = MS_SSIM(data_range=1, size_average=True, channel=1)
    loss = 1 - MS_SSIMfn(predicted, groundtruth)
    return loss.item()

## Configuration

In [3]:

saveimgs = False
label = "BGfunctionpixsplit_apply3x_Alltestdata"

base_path = r"C:\Users\ggjh246\OneDrive - University of Leeds\Code\ApplyModels_Eddie"
test_filename = "data"
test_path = os.path.join(base_path, test_filename)

AFM = np.load(r"C:\Users\ggjh246\OneDrive - University of Leeds\Code\afmlevel\src\afmlevel\lutAFM.npy")
AFM = ListedColormap(AFM)

MSEscore = False
SSIMscore = True
PSNRscore = True

if saveimgs:
    os.makedirs(f"{label}_images", exist_ok=True)


## Load Image IDs

In [4]:

imageidlist = []

for filename in os.listdir(test_path):
    if filename.endswith(".tiff") and not filename.endswith("_levelled.tiff"):
        imageidlist.append(filename[:-5])  # remove .tiff

imageidlist[:10]  # preview


['cal071504.008_1']

## Processing Loop

In [5]:
MSElist = []
SSIMlist = []
PSNRlist = []

for i, image_id in enumerate(imageidlist):

    print(f"Processing image {i+1}/{len(imageidlist)}: {image_id}")

    # --- Load image ---
    image = np.array(Image.open(os.path.join(test_path, f"{image_id}.tiff")))
    
    # --- Apply background model (1x, 2x, 3x) ---
    model_bg, model_levarray = afml.applymodel_bg_pixelsplit_all(image, 3)
    model_levnorm = znorm(model_levarray).astype(np.float32)

    model_bg2, model_levarray2 = afml.applymodel_bg_pixelsplit_all(model_levarray, 3)
    model_levnorm2 = znorm(model_levarray2)

    model_bg3, model_levarray3 = afml.applymodel_bg_pixelsplit_all(model_levarray2, 3)
    model_levnorm3 = znorm(model_levarray3)


Processing image 1/1: cal071504.008_1
unetmodel_Aire_BG_60eps_b32_d0_1f9_f9_7l_MSE_0line_train34_epoch59.pth


FileNotFoundError: Model file not found: unetmodel_Aire_BG_60eps_b32_d0_1f9_f9_7l_MSE_0line_train34_epoch59.pth

## Plotting

In [ ]:

plt.figure(figsize=(18, 4))
plt.subplot(1, 3, 1)
plt.title("Original")
plt.imshow(image, cmap=AFM)


plt.subplot(1, 3, 3)
plt.title("Model Levelled (first pass)")
plt.imshow(model_levnorm, cmap=AFM)

plt.show()


## Save Matrix

In [ ]:

import pandas as pd

df_ssim = pd.DataFrame(SSIMlist)
df_psnr = pd.DataFrame(PSNRlist)

df_ssim.head(), df_psnr.head()
